In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

## Description

|Variable|Definition   | Key  |  Type |
|---|---|---|---|
|id | Unique identifier for each tourist |
|country | The country a tourist coming  from.|
|age_group | The age group of a tourist.|
|travel_with | The relation of people a tourist travel with to Tanzania|
|total_female | Total number of females|
|total_male | Total number of males|
|purpose | The purpose of visiting  Tanzania|
|main_activity | The main activity of tourism in Tanzania|
|infor_source | The source of information about tourism in Tanzania|
|tour_arrangment | The arrangment of visiting Tanzania|
|package_transport_int | If the tour package include international transportation service|
|package_accomodation | If the tour package include accommodation service|
|package_food | If the tour package include food service|
|package_transport_tz | If the tour package include transport service within Tanzania| 
|package_sightseeing | If the tour package include sightseeing service|
|package_guided_tour | If the tour package include tour guide|
|package_insurance | if the tour package include insurance service|
|night_mainland | Number of nights a tourist spent in Tanzania mainland|
|night_zanzibar | Number of nights a tourist spent in Zanzibar|
|payment_mode | The mode of payment for tourism service|
|first_trip_tz | If it was a first  trip to Tanzania|
|most_impressing | what impressed a toursit in Tanzania|
|total_cost | The total tourist expenditure  in TZS(currency)|

In [2]:
df = pd.read_csv('data/train.csv', delimiter=',')

In [3]:
df["travel_with"] = df["travel_with"].fillna("Alone")
df["total_female"] = df["total_female"].fillna(0)
df["total_male"] = df["total_male"].fillna(0)
df["total_female"] = df["total_female"].astype(int)
df["total_male"] = df["total_male"].astype(int)
df[df["most_impressing"].isna()]
df.loc[df["most_impressing"].isna(), "most_impressing"] = "No comments"
df.drop(["ID"], axis=1, inplace=True)
df.reset_index(drop=True, inplace=True)


In [4]:
cols = df.select_dtypes(include=["object", "string"]).columns
df[cols] = df[cols].replace({
    "yes": 1,
    "no": 0,
    "Yes": 1,
    "No": 0
})

# Feature engineering

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 22 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   country                4809 non-null   object 
 1   age_group              4809 non-null   object 
 2   travel_with            4809 non-null   object 
 3   total_female           4809 non-null   int32  
 4   total_male             4809 non-null   int32  
 5   purpose                4809 non-null   object 
 6   main_activity          4809 non-null   object 
 7   info_source            4809 non-null   object 
 8   tour_arrangement       4809 non-null   object 
 9   package_transport_int  4809 non-null   int64  
 10  package_accomodation   4809 non-null   int64  
 11  package_food           4809 non-null   int64  
 12  package_transport_tz   4809 non-null   int64  
 13  package_sightseeing    4809 non-null   int64  
 14  package_guided_tour    4809 non-null   int64  
 15  pack

## Pipeline

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

X = df.drop("total_cost", axis=1)
y = df["total_cost"]


## Split cat and num cols

In [7]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object", "string"]).columns

## Preprocessing Pipelines

In [8]:
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [9]:
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

## Column Transformer

In [10]:
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

## Final Pipeline

In [11]:
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        random_state=42
    ))
])

## Splitting data for testing 

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train

In [13]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['package_transport_int', 'package_accomodation', 'package_food',
       'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
       'package_insurance', 'night_mainland', 'night_zanziba...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['country', 'age_group', 'travel_with', 'purpose', 'main_activity',
       'info_source', 'tour_arrangement', 'payment_mode', 'most_impressing'],
      dtype='object'))])),
                ('model',
                 RandomForestRegressor(max_depth=10, n_estimators=300,
                                       random_state=42))])

## Eval

In [14]:
from sklearn.metrics import r2_score, mean_absolute_error

y_pred = pipeline.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))

R2: 0.4006343635775098
MAE: 4846034.002087653


## Hyperparameter Tuning

In [15]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [5, 10, 20]
}

grid = GridSearchCV(pipeline, param_grid, cv=5, scoring="r2")
grid.fit(X_train, y_train)

print(grid.best_params_)

{'model__max_depth': 5, 'model__n_estimators': 300}


## Check performance

In [16]:
from sklearn.metrics import r2_score

y_pred = grid.best_estimator_.predict(X_test)
print("R2:", r2_score(y_test, y_pred))

R2: 0.3899898816767029


## Get Feature Importance from Pipeline

In [17]:
model = grid.best_estimator_.named_steps["model"]
importances = model.feature_importances_

## Feature Engineering

map purpose in 3 over categories

In [18]:
df["purpose_grouped"] = df["purpose"].replace({
    "Leisure and Holidays": "Leisure",
    "Visiting Friends and Relatives": "Leisure",
    
    "Business": "Work",
    "Meetings and Conference": "Work",
    "Scientific and Academic": "Work",
    
    "Volunteering": "Other",
    "Other": "Other"
})

In [19]:
df = df.drop("purpose", axis=1)

map main activity

In [20]:
df["main_activity_grouped"] = df["main_activity"].replace({
    "Wildlife tourism": "Wildlife",
    "Beach tourism": "Beach",
    "Hunting tourism": "Adventure",
    "Mountain climbing": "Adventure",
    "Cultural tourism": "Cultural",
    "Conference tourism": "Work",
    "business": "Work",
    "Bird watching": "Other",
    "Diving and Sport Fishing": "Other"
})

df = df.drop("main_activity", axis=1)

map most impressing

In [21]:
df["most_impressing"] = df["most_impressing"].str.strip()
df["most_impressing_grouped"] = df["most_impressing"].replace({
    "Friendly People": "People",
    "No comments": "NoComments",
    " Wildlife": "Wildlife",  # Achtung: führendes Leerzeichen entfernen!
    "Wonderful Country, Landscape, Nature": "Nature",
    "Good service": "Other",
    "Excellent Experience": "Other",
    "Satisfies and Hope Come Back": "Other"
})

df = df.drop("most_impressing", axis=1)

In [22]:
#df = df.drop("payment_mode", axis=1)
df

,country,age_group,travel_with,total_female,total_male,info_source,tour_arrangement,package_transport_int,package_accomodation,package_food,...,package_guided_tour,package_insurance,night_mainland,night_zanzibar,payment_mode,first_trip_tz,total_cost,purpose_grouped,main_activity_grouped,most_impressing_grouped
0,SWIZERLAND,45-64,Friends/Relatives,1,1,"Friends, relatives",Independent,0,0,0,...,0,0,13.0,0.0,Cash,0,674602.5,Leisure,Wildlife,People
1,UNITED KINGDOM,25-44,Alone,1,0,others,Independent,0,0,0,...,0,0,14.0,7.0,Cash,1,3214906.5,Leisure,Cultural,Nature
2,UNITED KINGDOM,25-44,Alone,0,1,"Friends, relatives",Independent,0,0,0,...,0,0,1.0,31.0,Cash,0,3315000.0,Leisure,Cultural,Other
3,UNITED KINGDOM,25-44,Spouse,1,1,"Travel, agent, tour operator",Package Tour,0,1,1,...,1,0,11.0,0.0,Cash,1,7790250.0,Leisure,Wildlife,People
4,CHINA,1-24,Alone,1,0,"Travel, agent, tour operator",Independent,0,0,0,...,0,0,7.0,4.0,Cash,1,1657500.0,Leisure,Wildlife,NoComments
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4804,UAE,45-64,Alone,0,1,"Friends, relatives",Independent,0,0,0,...,0,0,2.0,0.0,Credit Card,0,3315000.0,Work,Adventure,NoComments
4805,UNITED STATES OF AMERICA,25-44,Spouse,1,1,"Travel, agent, tour operator",Package Tour,1,1,1,...,1,1,11.0,0.0,Cash,1,10690875.0,Leisure,Wildlife,People
4806,NETHERLANDS,1-24,Alone,1,0,others,Independent,0,0,0,...,0,0,3.0,7.0,Cash,1,2246636.7,Leisure,Wildlife,Other
4807,SOUTH AFRICA,25-44,Friends/Relatives,1,1,"Travel, agent, tour operator",Independent,1,1,1,...,0,0,5.0,0.0,Credit Card,0,1160250.0,Work,Beach,People


Define the whole Pipeline Process in a function so we can run it again

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import pandas as pd

def train_random_forest_pipeline(df, target="total_cost", test_size=0.2, random_state=42,
                                 n_estimators=500, max_depth=10):
    """
    Führt alles in einem Schritt aus:
    - Feature/Target split
    - Train/Test split
    - Preprocessing (numerisch + kategorisch)
    - Random Forest trainieren
    - Evaluation (R², MAE)
    
    Returns: pipeline, X_train, X_test, y_train, y_test, y_pred
    """
    # 1️⃣ Features und Target
    X = df.drop(target, axis=1)
    y = df[target]
    
    # 2️⃣ Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # 3️⃣ Spaltentypen erkennen
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object", "string"]).columns
    
    # 4️⃣ Pipelines für numerische + kategorische Features
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])
    
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])
    
    # 5️⃣ Ganze Pipeline
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features="sqrt",
            n_jobs=-1
        ))
    ])
    
    # 6️⃣ Trainieren
    pipeline.fit(X_train, y_train)
    
    # 7️⃣ Vorhersage & Evaluation
    y_pred = pipeline.predict(X_test)
    print(f"R2: {r2_score(y_test, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_test, y_pred):.2f}")
    
    return pipeline, X_train, X_test, y_train, y_test, y_pred

In [24]:
pipeline, X_train, X_test, y_train, y_test, y_pred = train_random_forest_pipeline(df)

R2: 0.391
MAE: 5102135.22


## Feature Engineering

In [25]:
df["total_people"] = df["total_female"] + df["total_male"]
df["female_ratio"] = df["total_female"] / df["total_people"]
df["male_ratio"] = df["total_male"] / df["total_people"]
df["total_nights"] = df["night_mainland"] + df["night_zanzibar"]
df["package_count"] = df["package_transport_int"] +df["package_accomodation"] + df["package_food"] + df["package_transport_tz"] + df["package_sightseeing"] + df["package_guided_tour"] + df["package_insurance"]

In [26]:
pipeline, X_train, X_test, y_train, y_test, y_pred = train_random_forest_pipeline(df)

R2: 0.399
MAE: 5035521.96


In [27]:
""" payment_mode = df["payment_mode"]
df = df.drop("payment_mode", axis=1) """
# wieder enablen für rerun 


' payment_mode = df["payment_mode"]\ndf = df.drop("payment_mode", axis=1) '

In [28]:
pipeline, X_train, X_test, y_train, y_test, y_pred = train_random_forest_pipeline(df)

R2: 0.399
MAE: 5035521.96


## Boosting Pipeline

In [29]:
df_boosting = pd.read_csv('data/train.csv', delimiter=',')

In [30]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np
import pandas as pd

def train_boosting_pipeline(df, target="total_cost", test_size=0.2,
                            random_state=42, log_transform=True,
                            max_iter=500, max_depth=6, learning_rate=0.05):
    """
    Pipeline:
    - Feature Engineering: neue Features + Gruppierung von Kategorien
    - Preprocessing: numerische + kategorische Pipelines
    - HistGradientBoostingRegressor trainieren
    - Evaluation (R2, MAE) auf Originalscale
    """
    
    df = df.copy()
    
    # 2️⃣ Features & Target
    X = df.drop(target, axis=1)
    y = df[target].copy()
    
    # 3️⃣ Log-Transformation optional
    if log_transform:
        y = np.log1p(y)
    
    # 4️⃣ Train/Test Split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )
    
    # 5️⃣ Spaltentypen
    num_cols = X.select_dtypes(include=["int64", "float64"]).columns
    cat_cols = X.select_dtypes(include=["object", "string"]).columns
    
    # 6️⃣ Pipelines
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])
    
    preprocessor = ColumnTransformer([
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ])
    
    # 7️⃣ Boosting Pipeline
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", HistGradientBoostingRegressor(
            max_iter=max_iter,
            max_depth=max_depth,
            learning_rate=learning_rate,
            random_state=random_state
        ))
    ])
    
    # 8️⃣ Trainieren
    pipeline.fit(X_train, y_train)
    
    # 9️⃣ Vorhersage
    y_pred_log = pipeline.predict(X_test)
    if log_transform:
        y_pred = np.expm1(y_pred_log)
        y_test_real = np.expm1(y_test)
    else:
        y_pred = y_pred_log
        y_test_real = y_test
    
    # 10️⃣ Evaluation
    print(f"R2: {r2_score(y_test_real, y_pred):.3f}")
    print(f"MAE: {mean_absolute_error(y_test_real, y_pred):.2f}")
    
    return pipeline, X_train, X_test, y_train, y_test, y_pred

little cleaning again

In [31]:
df_boosting["travel_with"] = df["travel_with"].fillna("Alone")
df_boosting["total_female"] = df["total_female"].fillna(0)
df_boosting["total_male"] = df["total_male"].fillna(0)
df_boosting["total_female"] = df_boosting["total_female"].astype(int)
df_boosting["total_male"] = df_boosting["total_male"].astype(int)
df_boosting[df_boosting["most_impressing"].isna()]
df_boosting.loc[df_boosting["most_impressing"].isna(), "most_impressing"] = "No comments"
df_boosting.drop(["ID"], axis=1, inplace=True)
df_boosting.reset_index(drop=True, inplace=True)

In [32]:
cols = df_boosting.select_dtypes(include=["object", "string"]).columns
df_boosting[cols] = df_boosting[cols].replace({
    "yes": 1,
    "no": 0,
    "Yes": 1,
    "No": 0
})

In [33]:
# Alle Inf durch NaN ersetzen
df_boosting.replace([np.inf, -np.inf], np.nan, inplace=True)

# NaN durch Median (oder 0) ersetzen
df_boosting.fillna(df_boosting.median(numeric_only=True), inplace=True)

In [34]:

pipeline, X_train, X_test, y_train, y_test, y_pred = train_boosting_pipeline(df)

R2: 0.340
MAE: 4791657.80


## Evaluate the model in a function

In [35]:
def evaluate_model(y_test, y_pred):
    from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    
    print(f"R2: {r2:.3f}")
    print(f"MAE: {mae:.2f}")
    print(f"MSE: {mse:.2f}")
    
    return {"R2": r2, "MAE": mae, "MSE": mse}

## improve the baseline Random forest Model

In [36]:
#n_estimators=500 → more stable
#max_depth=10 → more expressive than 5
#min_samples_leaf=2 → smoother predictions
#max_features="sqrt" → reduces overfitting
pipeline, X_train, X_test, y_train, y_test, y_pred = train_random_forest_pipeline(df,n_estimators=500, max_depth=10)

R2: 0.399
MAE: 5035521.96


# Proper Hyperparameter Tuning

In [37]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

param_dist = {
    "model__n_estimators": [200, 300, 500, 800],
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2"]
}

search = RandomizedSearchCV(
    pipeline,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)

Best params: {'model__n_estimators': 500, 'model__min_samples_split': 5, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': None}


## Evaluate best model

In [38]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

evaluate_model(y_test, y_pred)

R2: 0.417
MAE: 4856901.06
MSE: 82379521094925.20


{'R2': 0.41657472367080095, 'MAE': 4856901.055329235, 'MSE': 82379521094925.2}

## Feature Importance

In [39]:

def get_important_original_features(model, threshold=0.01):
    preprocessor = model.named_steps["preprocessing"]
    rf_model = model.named_steps["model"]
    
    feature_names = preprocessor.get_feature_names_out()
    importances = rf_model.feature_importances_
    
    import pandas as pd
    feat_imp = pd.Series(importances, index=feature_names)
    
    # Mapping zurück zu Original-Features
    important_original = set()
    
    for feat, importance in feat_imp.items():
        if importance > threshold:
            # Beispiel: "cat__country_USA" → "country"
            original = feat.split("__")[1].split("_")[0]
            important_original.add(original)
    
    return list(important_original), feat_imp.sort_values(ascending=False)

## Reduce the dataset

In [40]:
important_cols, feat_imp = get_important_original_features(best_model)

print("Wichtige Original Features:", important_cols)

# Nur diese behalten
#df_reduced = df[important_cols + ["total_cost"]]
feat_imp
##here we can see to drop country is good for our model
#TODO: drop country from df and retrain model to see if performance improves
#df = df.drop("country", axis=1)
pipeline, X_train, X_test, y_train, y_test, y_pred = train_random_forest_pipeline(df,n_estimators=500, max_depth=10)
#R2: 0.376
#MAE: 5152572.08
#es wurde schlechterweise nicht besser, sondern schlechter... vielleicht weil wir zu wenige Features haben? oder weil country doch wichtig ist? oder weil wir zu wenig tunen? oder weil RF einfach nicht so gut ist?

Wichtige Original Features: ['total', 'payment', 'male', 'info', 'most', 'country', 'main', 'first', 'age', 'package', 'female', 'night', 'tour', 'travel']
R2: 0.399
MAE: 5035521.96
